# Multi Label Approach

So far we have been experimenting with multi-class classification (model classifies each survey response into ONE category only) and we managed to approximately reach the 80% threshold.

But now, we want to experiment with a multi-label approach where a model can classify each survey response into mulitple categories and usually assigns a metric like confidence for each category indicating how confident it is that the response belongs to the categories it has classified the response as.

### Setting Seeds

In [1]:
import random
from transformers import set_seed
import os

import pandas as pd
import numpy as np
import warnings
import ast
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support, multilabel_confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import torch
import torch.nn as nn
from transformers import TrainingArguments, Trainer, EarlyStoppingCallback
from datasets import Dataset

import warnings
warnings.filterwarnings('ignore') 

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# Set all seeds
set_all_seeds(42)

### Data Preprocessing

In [2]:
# Import clean survey data
clean_df = pd.read_excel('synthetic_data_mixed_labels.xlsx')

# Preview
pd.set_option('display.max_colwidth', 50)
print(clean_df.shape)
clean_df.head(3)

(675, 16)


,course_name,question,student_response,inappropriate_appropriate,category,category_mixed,all_labels,primary_labels,secondary_labels,primary_labels_str,secondary_labels_str,sim_teaching & delivery,sim_assignment & quiz,sim_course materials & structure,num_labels,clean_text
0,Introduction to Computer Science,Which aspects of this course need improvement?,the lecture recordings are often unclear and t...,appropriate,teaching & delivery,teaching & delivery,['teaching & delivery'],['teaching & delivery'],[],teaching & delivery,NaN,0.569394,0.551357,0.526510,1,lecture recording often unclear audio quality ...
1,Mathematical Methods,Which aspects of this course need improvement?,this course sucks and prof smith is the worst ...,inappropriate,teaching & delivery,mixed,"['teaching & delivery', 'course materials & st...",['teaching & delivery'],['course materials & structure'],teaching & delivery,course materials & structure,0.735655,0.625059,0.721436,2,course suck prof smith bad teacher ever comple...
2,Physics for Engineers,Which aspects of this course need improvement?,the assignment instructions are very ambiguous...,appropriate,assignment & quiz,mixed,"['assignment & quiz', 'course materials & stru...",['assignment & quiz'],['course materials & structure'],assignment & quiz,course materials & structure,0.589858,0.659662,0.618404,2,assignment instruction ambiguous due date clos...


In [3]:
# Sort each list alphabetically
clean_df['all_labels'] = clean_df['all_labels'].apply(lambda x: sorted(x) if isinstance(x, list) else x)
# testing with 'clean_text'
filtered_df = clean_df[clean_df['student_response'].str.len()>10].copy()

In [4]:
import ast
import pandas as pd

# Clean the label
def clean_and_sort_labels(x):
    # Convert stringified list or split comma-separated strings
    if isinstance(x, str):
        x = x.strip()
        if x.startswith("[") and x.endswith("]"):
            try:
                x_list = ast.literal_eval(x)
            except:
                # fallback: split inside brackets
                x_list = x.strip("[]").split(",")
        else:
            x_list = x.split(",")
    elif isinstance(x, list):
        x_list = x
    else:
        return []

    # Clean each item: strip spaces, remove stray brackets/quotes
    clean_list = []
    for item in x_list:
        if not isinstance(item, str):
            item = str(item)
        item = item.strip().strip("[]\"'")
        if item:  # skip empty strings
            clean_list.append(item)
    
    # Deduplicate and sort alphabetically (case-insensitive)
    seen = set()
    final_list = []
    for i in sorted(clean_list, key=lambda s: s.lower()):
        if i not in seen:
            seen.add(i)
            final_list.append(i)
    
    return final_list

# Apply to your dataframe
filtered_df['all_labels'] = filtered_df['all_labels'].apply(clean_and_sort_labels)

In [5]:
filtered_df['all_labels'].value_counts()

all_labels
[course materials & structure]                                            172
[teaching & delivery]                                                     154
[assignment & quiz]                                                       146
[assignment & quiz, course materials & structure, teaching & delivery]     89
[course materials & structure, teaching & delivery]                        53
[assignment & quiz, course materials & structure]                          36
[assignment & quiz, teaching & delivery]                                   25
Name: count, dtype: int64

In [6]:
from sklearn.preprocessing import MultiLabelBinarizer

### Encode the labels
mlb = MultiLabelBinarizer(classes=["course materials & structure", "teaching & delivery", "assignment & quiz"])
filtered_df["category_multi"] = filtered_df["all_labels"]  # assume each row has a list of labels
y_multi = mlb.fit_transform(filtered_df["category_multi"])

app_map = {"inappropriate": 0, "appropriate": 1}
filtered_df['inappropriate_appropriate'] = filtered_df['inappropriate_appropriate'].map(app_map)
y_binary = filtered_df['inappropriate_appropriate'].copy()

### Split The Data

In [7]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Combine both label types into a single multi-label matrix
combined_labels = np.column_stack([y_binary.values.reshape(-1, 1), y_multi])
X = filtered_df['student_response'].copy()

msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.1, random_state=42)
train_idx, test_idx = next(msss.split(X, combined_labels))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_binary_train, y_binary_test = y_binary.iloc[train_idx], y_binary.iloc[test_idx]
y_multi_train, y_multi_test = y_multi[train_idx], y_multi[test_idx]

In [8]:
from collections import Counter

print(f'Initial shape: {clean_df.shape}')
print(f'Filtered shape: {filtered_df.shape}\n')

print(f'Train proportion label 1: {Counter(y_binary_train)}')
y_multi_train_str = ['_'.join(map(str, row)) for row in y_multi_train]
print(f'Train proportion label 2: \n{pd.Series(y_multi_train_str).value_counts()}\n')

print(f'Test proportion label 1: {Counter(y_binary_test)}')
y_multi_test_str = ['_'.join(map(str, row)) for row in y_multi_test]
print(f'Test proportion label 2: \n{pd.Series(y_multi_test_str).value_counts()}\n')

Initial shape: (675, 16)
Filtered shape: (675, 17)

Train proportion label 1: Counter({1: 455, 0: 155})
Train proportion label 2: 
1_0_0    153
0_1_0    142
0_0_1    136
1_1_1     76
1_1_0     49
1_0_1     32
0_1_1     22
Name: count, dtype: int64

Test proportion label 1: Counter({1: 51, 0: 14})
Test proportion label 2: 
1_0_0    19
1_1_1    13
0_1_0    12
0_0_1    10
1_1_0     4
1_0_1     4
0_1_1     3
Name: count, dtype: int64



In [9]:
train_df = pd.DataFrame({
    "student_response": X_train.values
})

test_df = pd.DataFrame({
    "student_response": X_test.values
})

train_df.labels = y_multi_train
test_df.labels = y_multi_test

In [10]:
train_df.labels

array([[0, 1, 0],
       [1, 0, 1],
       [1, 0, 0],
       ...,
       [1, 0, 0],
       [0, 1, 0],
       [0, 0, 1]])

### Loading and Pre-Processing Test Data

In [11]:
# Load your separate test set
test_set_df = pd.read_excel('synthetic_data_test.xlsx')  # Replace with your actual test file name

# Apply the same filtering (responses longer than 10 characters)
test_filtered_df = test_set_df[test_set_df['student_response'].str.len() > 10].copy()

# Apply the same label cleaning and sorting function
test_filtered_df['all_labels'] = test_filtered_df['all_labels'].apply(clean_and_sort_labels)

# IMPORTANT: Use the already fitted MultiLabelBinarizer from training data
# Do NOT fit a new one - use transform only
y_multi_test_final = mlb.transform(test_filtered_df['all_labels'])

# Apply the same mapping for binary labels
test_filtered_df['inappropriate_appropriate'] = test_filtered_df['inappropriate_appropriate'].map(app_map)
y_binary_test_final = test_filtered_df['inappropriate_appropriate'].copy()

# Create final test dataframe in same format
final_test_df = pd.DataFrame({
    "student_response": test_filtered_df['student_response'].values
})

# Attach labels as attributes (matching your training format)
final_test_df.labels = y_multi_test_final

print(f"Test set shape: {final_test_df.shape}")
print(f"Multi-label shape: {y_multi_test_final.shape}")
print(f"Binary label shape: {y_binary_test_final.shape}")

# Verify the label distributions
print("\nMulti-label distribution in test set:")
for i, label in enumerate(mlb.classes_):
    print(f"{label}: {y_multi_test_final[:, i].sum()} samples")

print(f"\nBinary label distribution in test set:")
print(f"Inappropriate (0): {(y_binary_test_final == 0).sum()}")
print(f"Appropriate (1): {(y_binary_test_final == 1).sum()}")

Test set shape: (104, 1)
Multi-label shape: (104, 3)
Binary label shape: (104,)

Multi-label distribution in test set:
course materials & structure: 48 samples
teaching & delivery: 61 samples
assignment & quiz: 43 samples

Binary label distribution in test set:
Inappropriate (0): 31
Appropriate (1): 73


In [12]:
# Check the DataFrame structure
print("DataFrame columns:", train_df.columns.tolist())
print("DataFrame shape:", train_df.shape)

# Check the labels attribute
print("Labels shape:", train_df.labels.shape)
print("Labels type:", type(train_df.labels))

# See a few examples
print("\nFirst 5 labels:")
print(train_df.labels[:5])

# Check if labels match the number of samples
print(f"\nDataFrame has {len(train_df)} rows")
print(f"Labels array has {len(train_df.labels)} rows")

DataFrame columns: ['student_response']
DataFrame shape: (610, 1)
Labels shape: (610, 3)
Labels type: <class 'numpy.ndarray'>

First 5 labels:
[[0 1 0]
 [1 0 1]
 [1 0 0]
 [1 0 0]
 [1 0 0]]

DataFrame has 610 rows
Labels array has 610 rows


### Load Model and Tokeniser

In [13]:
# Model: DeBERTa-v3-base (light but strong) microsoft/deberta-v3-base
MODEL_NAME = "microsoft/deberta-v3-base"

# Loading Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

multi_label_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=3,
    problem_type="multi_label_classification")

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
# Ensure model runs on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
multi_label_model.to(device)

DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): Layer

### Layer Freezing

In [15]:
def freeze_lower_layers(model, num_layers_to_freeze=8):
    # Freeze embedding and first N encoder layers
    for param in model.deberta.embeddings.parameters():
        param.requires_grad = False
    
    for i in range(num_layers_to_freeze):
        for param in model.deberta.encoder.layer[i].parameters():
            param.requires_grad = False

    print(f"Frozen first {num_layers_to_freeze} layers and embeddings")
    
    # Print number of trainable parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Frozen parameters: {total_params - trainable_params:,}")

freeze_lower_layers(multi_label_model, num_layers_to_freeze=8)

Frozen first 8 layers and embeddings
Total parameters: 184,424,451
Trainable parameters: 29,339,139
Frozen parameters: 155,085,312


### Create Hugging Face Datasets and Tokenise

In [16]:
train_dataset = Dataset.from_pandas(train_df)
test_dataset = Dataset.from_pandas(test_df)
final_test_dataset = Dataset.from_pandas(final_test_df)

In [17]:
print(train_dataset)
print(test_dataset)
print(final_test_dataset)

Dataset({
    features: ['student_response'],
    num_rows: 610
})
Dataset({
    features: ['student_response'],
    num_rows: 65
})
Dataset({
    features: ['student_response'],
    num_rows: 104
})


In [18]:
train_dataset = train_dataset.add_column("labels", train_df.labels.tolist())
test_dataset = test_dataset.add_column("labels", test_df.labels.tolist())
final_test_dataset = final_test_dataset.add_column("labels", final_test_df.labels.tolist())

In [19]:
def tokenize(batch):
    texts = [str(x) for x in batch["student_response"]]
    return tokenizer(texts, 
                     padding="max_length", 
                     truncation=True, 
                     max_length=256)

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)
final_test_dataset = final_test_dataset.map(tokenize, batched=True)

Map:   0%|          | 0/610 [00:00<?, ? examples/s]

Map:   0%|          | 0/65 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

In [20]:
print(train_dataset)
print(test_dataset)
print(final_test_dataset)

Dataset({
    features: ['student_response', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 610
})
Dataset({
    features: ['student_response', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 65
})
Dataset({
    features: ['student_response', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 104
})


In [21]:
# Remove unwanted columns
columns_to_remove = [col for col in train_dataset.column_names 
                    if col not in ["input_ids", "attention_mask", "labels", "student_response"]]

# This will remove token_type_ids along with other unwanted columns
train_dataset = train_dataset.remove_columns(columns_to_remove)
test_dataset = test_dataset.remove_columns(columns_to_remove)
final_test_dataset = final_test_dataset.remove_columns(columns_to_remove)

In [22]:
print(train_dataset)
print(test_dataset)
print(final_test_dataset)

Dataset({
    features: ['student_response', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 610
})
Dataset({
    features: ['student_response', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 65
})
Dataset({
    features: ['student_response', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 104
})


In [23]:
from datasets import Sequence, Value

train_dataset = train_dataset.cast_column("labels", Sequence(Value("float32")))
test_dataset = test_dataset.cast_column("labels", Sequence(Value("float32")))
final_test_dataset = final_test_dataset.cast_column("labels", Sequence(Value("float32")))

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
final_test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Casting the dataset:   0%|          | 0/610 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/65 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/104 [00:00<?, ? examples/s]

In [24]:
print(train_dataset[0]["labels"])          # tensor([...], dtype=torch.float32)
print(train_dataset[0]["labels"].dtype)    # torch.float32

tensor([0., 1., 0.])
torch.float32


### Define Metrics

In [25]:
def compute_metrics_multilabel(eval_pred):
    logits, labels = eval_pred
    
    # For multi-label: apply sigmoid and threshold at 0.5
    predictions = torch.sigmoid(torch.from_numpy(logits))
    predictions = (predictions > 0.5).float().numpy()
    
    # Calculate metrics for each label and average
    f1_micro = f1_score(labels, predictions, average='micro')
    f1_macro = f1_score(labels, predictions, average='macro')
    f1_weighted = f1_score(labels, predictions, average='weighted')
    
    # Subset accuracy (exact match - all labels correct)
    subset_accuracy = (predictions == labels).all(axis=1).mean()
    
    # Hamming loss (fraction of wrong labels)
    hamming_loss = (predictions != labels).mean()
    
    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro, 
        "f1_weighted": f1_weighted,
        "subset_accuracy": subset_accuracy,
        "hamming_loss": hamming_loss
    }

### Implement Class Weighting

In [26]:
class MultiLabelWeightedTrainer(Trainer):
    def __init__(self, class_weights=None, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
    
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        
        
        # Use BCEWithLogitsLoss for multi-label (includes sigmoid)
        pos_w = None
        if self.class_weights is not None:
            pos_w = self.class_weights.to(model.device).to(logits.dtype)

        
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_w)
        loss = loss_fct(logits, labels.float())
        
        return (loss, outputs) if return_outputs else loss

# Calculate class weights for multi-label
def calculate_multilabel_class_weights(labels_array):
    """Calculate positive weights for each class in multi-label setting"""
    pos_weights = []
    n_samples = len(labels_array)
    
    for i in range(labels_array.shape[1]):  # For each class
        pos_count = labels_array[:, i].sum()
        neg_count = n_samples - pos_count
        
        if pos_count > 0:
            pos_weight = neg_count / pos_count
        else:
            pos_weight = 1.0
            
        pos_weights.append(pos_weight)
    
    return torch.FloatTensor(pos_weights).to(device)

# Calculate class weights
class_weights = calculate_multilabel_class_weights(train_df.labels)

print("Multi-label class weights (pos_weight):", class_weights)
print("This means:")
labels_names = ["assignment & quiz", "course materials & structure", "teaching & delivery"]
for i, name in enumerate(labels_names):
    print(f"  {name}: {class_weights[i]:.3f}")

# Print label distribution
print("\nLabel distribution in training data:")
for i, name in enumerate(labels_names):
    count = train_df.labels[:, i].sum()
    percentage = (count / len(train_df.labels)) * 100
    print(f"  {name}: {count} samples ({percentage:.1f}%)")

Multi-label class weights (pos_weight): tensor([0.9677, 1.1107, 1.2932], device='cuda:0')
This means:
  assignment & quiz: 0.968
  course materials & structure: 1.111
  teaching & delivery: 1.293

Label distribution in training data:
  assignment & quiz: 310 samples (50.8%)
  course materials & structure: 289 samples (47.4%)
  teaching & delivery: 266 samples (43.6%)


### Training Setup

In [27]:
from transformers.data.data_collator import default_data_collator
# Training arguments
training_args = TrainingArguments(
    output_dir="./results_multi_label",
    eval_strategy="epoch",
    save_strategy="epoch", 
    logging_strategy="epoch",
    
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1_macro",  # Use macro F1 for multi-label
    greater_is_better=True,

    learning_rate=2e-5,  # Start with slightly lower LR for multi-label
    per_device_eval_batch_size=8,
    per_device_train_batch_size=8,
    num_train_epochs=12,
    weight_decay=0.01,
    gradient_accumulation_steps=2,
    warmup_ratio=0.1,  # Use ratio instead of steps

    fp16=True,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    
    logging_dir="./logs_multi_label",
    seed=42,
)

# Early stopping
early_stopping_callback = EarlyStoppingCallback(
    early_stopping_patience=3,
    early_stopping_threshold=0.003
)

def float_labels_collator(features):
    batch = default_data_collator(features)
    # ensure float32 (Trainer may autocast to fp16 as needed)
    batch["labels"] = batch["labels"].to(torch.float32)
    return batch

# Create trainer
trainer = MultiLabelWeightedTrainer(
    model=multi_label_model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics_multilabel,
    callbacks=[early_stopping_callback],
    class_weights=class_weights,
    data_collator=float_labels_collator,
)

### Train Model

In [28]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro,F1 Weighted,Subset Accuracy,Hamming Loss,Runtime,Samples Per Second,Steps Per Second
1,0.730700,0.730839,0.523810,0.460402,0.420078,0.184615,0.410256,0.462300,140.611000,19.469000
2,0.690300,0.568008,0.766990,0.765052,0.768488,0.415385,0.246154,0.425600,152.742000,21.149000
3,0.567600,0.522804,0.741463,0.739821,0.734949,0.430769,0.271795,0.428700,151.604000,20.991000
4,0.495000,0.465872,0.758294,0.763528,0.760886,0.492308,0.261538,0.441600,147.201000,20.382000
5,0.453000,0.430999,0.803922,0.806807,0.804783,0.523077,0.205128,0.422700,153.769000,21.291000
6,0.409900,0.389379,0.863850,0.862623,0.866451,0.661538,0.148718,0.422400,153.892000,21.308000
7,0.339900,0.342470,0.881188,0.879790,0.881347,0.753846,0.123077,0.442200,147.002000,20.354000
8,0.294600,0.312955,0.883495,0.881300,0.884047,0.738462,0.123077,0.422900,153.716000,21.284000
9,0.280100,0.307646,0.907317,0.906956,0.907377,0.800000,0.097436,0.429200,151.436000,20.968000
10,0.241000,0.304684,0.911765,0.911892,0.911732,0.815385,0.092308,0.423800,153.391000,21.239000


TrainOutput(global_step=468, training_loss=0.4134940876919999, metrics={'train_runtime': 174.4988, 'train_samples_per_second': 41.949, 'train_steps_per_second': 2.682, 'total_flos': 963012378931200.0, 'train_loss': 0.4134940876919999, 'epoch': 12.0})

### Evaluation

In [29]:
trainer.evaluate(final_test_dataset)

{'eval_loss': 0.5555571913719177,
 'eval_f1_micro': 0.7753623188405797,
 'eval_f1_macro': 0.7793555671356609,
 'eval_f1_weighted': 0.7715951633251968,
 'eval_subset_accuracy': 0.5480769230769231,
 'eval_hamming_loss': 0.1987179487179487,
 'eval_runtime': 0.9218,
 'eval_samples_per_second': 112.818,
 'eval_steps_per_second': 14.102,
 'epoch': 12.0}